# Demo 2: Controlled Pendulum
## Demo 2.6: Hybrid - Unified within SysSimX Framework

### Description

The following demo implements a controlled pendulum system described in Demo 2.1. The Modelica models for the `Reference`, `AngleEncoder`, `Controller`, and `Drive` are exported as an Co-Simulation FMU using OpenModelica. The FMUs are then imported and simulated in Python using the FMPy package.

The pendulum itself again modeled as an OpenSim model and an FEM Model. In the free swing phase (no contact with the wall), the pendulum is simulated using OpenSim. When the pendulum comes close to the wall the simulation switches to the FEM model to accurately capture the contact dynamics.
### Features of the `SysSimX` Framework

- The Framework supports the **configuration of FMUs within a YAML configuration file** where the path to the FMU file, inputs, and outputs can be specified
- The framework provides a **unified interface for working with co-simulation FMUs and OpenSim models**
- Both FMUs and OpenSim models follow the **CoSimComponent interface protocol**, allowing for seamless integration and interaction
- Similar methods for getting and setting variables, initialization and setup, and performing simulation steps


### Procedure

**1. Changing the working dirctory to use `SysSimX` package**

In [1]:
import sys
import numpy as np
#from pathlib import Path
from path import Path
from tabulate import tabulate
from fmpy import read_model_description

repo_root = Path.getcwd().parent.parent
sys.path.insert(0, str(repo_root))

In [2]:
from SysSimX.utilities.update_fmus import get_fmu_paths
demo_dir_path = Path(repo_root / 'demos' / 'ControlledPendulum')
package_path = Path(demo_dir_path / 'ControlledPendulum')
fmu_output_dir = Path(demo_dir_path / 'FMUs')

package_path.files("*.mo")

[Path('/home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/Demo_Driven.mo'),
 Path('/home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/Demo_UndrivenWithWall.mo'),
 Path('/home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/Drive.mo'),
 Path('/home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/PID_Continuous.mo'),
 Path('/home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/Pendulum.mo'),
 Path('/home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/ImpactWall.mo'),
 Path('/home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/package.mo'),
 Path('/home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/AngleEncoder.mo'),
 Path('/home/flo/repos/SystemSimulation/demos/ControlledPendulum/ControlledPendulum/Demo_DrivenWithWall.mo'),
 Path('/home/flo/repos/SystemSimulation/demos/ControlledPendulum/Controlle

In [3]:
# fmu_paths = get_fmu_paths(package_path, fmu_output_dir, force_rebuild=False)
# for fmu, path in fmu_paths.items():
#     print(f"{fmu:<25}: {path}")

In [4]:
from SysSimX.components.fmu_component import FMUComponent

def build_fmus(fmu_dir, fmu_names, fmu_component_links):
    fmus = {}
    for model_name, fmu_name in fmu_component_links.items():
        fmu_path = fmu_dir / f"{fmu_name}.fmu"
        md = read_model_description(fmu_path)
        fmu_model = FMUComponent(model_name, fmu_path)
        fmus[model_name] = fmu_model
    return fmus

In [5]:
fmu_dir   = repo_root / "demos" / "ControlledPendulum" / "FMUs"
fmu_names = ["Reference", "AngleEncoder", "PID_Continuous", "Drive2"]

fmu_component_links = {"ref": "Reference",
                       "sensor_ref": "AngleEncoder",
                       "sensor_state": "AngleEncoder",
                       "pid": "PID_Continuous",
                       "drive": "Drive2",
                       "pendulum": "Pendulum"}

fmu_models = build_fmus(fmu_dir, fmu_names, fmu_component_links)

ref          = fmu_models["ref"]
sensor_ref   = fmu_models["sensor_ref"]
sensor_state = fmu_models["sensor_state"]
pid          = fmu_models["pid"]
drive        = fmu_models["drive"]

LOG_SOLVER        | info    | CVODE linear multistep method CV_BDF
LOG_SOLVER        | info    | CVODE maximum integration order CV_ITER_NEWTON
LOG_SOLVER        | info    | CVODE use equidistant time grid YES
LOG_SOLVER        | info    | CVODE Using relative error tolerance 1.000000e-06
LOG_SOLVER        | info    | CVODE Using dense internal linear solver SUNLinSol_Dense.
LOG_SOLVER        | info    | CVODE Use internal dense numeric jacobian method.
LOG_SOLVER        | info    | CVODE uses internal root finding method NO
LOG_SOLVER        | info    | CVODE maximum absolut step size 0
LOG_SOLVER        | info    | CVODE initial step size is set automatically
LOG_SOLVER        | info    | CVODE maximum integration order 5
LOG_SOLVER        | info    | CVODE maximum number of nonlinear convergence failures permitted during one step 10
LOG_SOLVER        | info    | CVODE BDF stability limit detection algorithm OFF
LOG_SOLVER        | info    | CVODE linear multistep method CV_BDF
LOG_S

In [6]:
ref._md.description = "Sinusoidal Reference Trajectory"

ref_freq = 1 # Hz
q_mean_deg = 2.5
q_ampl_deg = -3
ref.parameters['mean'].start = np.deg2rad(q_mean_deg)
ref.parameters['amplitude'].start = np.deg2rad(q_ampl_deg)
ref.parameters['frequency'].start = ref_freq
print(ref)

FMU Name: ref
FMU Path: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/Reference.fmu
FMU Description: Sinusoidal Reference Trajectory

| Name      | Type   |      Start | Unit   | Causality   | Variability   |
|-----------|--------|------------|--------|-------------|---------------|
| q_ref     | Real   |            | rad    | output      | continuous    |
| amplitude | Real   | -0.0523599 | rad    | parameter   | fixed         |
| frequency | Real   |  1         | Hz     | parameter   | fixed         |
| mean      | Real   |  0.0436332 | rad    | parameter   | fixed         |



In [7]:
sensor_ref.parameters['q_max'].start = np.deg2rad(q_mean_deg + q_ampl_deg)
sensor_ref.parameters['q_min'].start = np.deg2rad(q_mean_deg - q_ampl_deg)
sensor_state.parameters['q_max'].start = np.deg2rad(q_mean_deg + q_ampl_deg)
sensor_state.parameters['q_min'].start = np.deg2rad(q_mean_deg - q_ampl_deg)

print(sensor_ref)
print(sensor_state)

FMU Name: sensor_ref
FMU Path: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/AngleEncoder.fmu
FMU Description: 

| Name   | Type   |       Start | Unit   | Causality   | Variability   |
|--------|--------|-------------|--------|-------------|---------------|
| U_q    | Real   |             | V      | output      | continuous    |
| q      | Real   |  0          | rad    | input       | continuous    |
| nBits  | Real   |  8          |        | parameter   | fixed         |
| q_max  | Real   | -0.00872665 | rad    | parameter   | fixed         |
| q_min  | Real   |  0.0959931  | rad    | parameter   | fixed         |

FMU Name: sensor_state
FMU Path: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/AngleEncoder.fmu
FMU Description: 

| Name   | Type   |       Start | Unit   | Causality   | Variability   |
|--------|--------|-------------|--------|-------------|---------------|
| U_q    | Real   |             | V      | output      | continuous    |
| q    

In [8]:
pid.parameters['k'].start = 20
pid.parameters['Ti'].start = 0.5
pid.parameters['Td'].start = 0.1
print(pid)

FMU Name: pid
FMU Path: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/PID_Continuous.fmu
FMU Description: 

| Name   | Type   |   Start | Unit   | Causality   | Variability   |
|--------|--------|---------|--------|-------------|---------------|
| ref    | Real   |     0   |        | input       | continuous    |
| u      | Real   |         |        | output      | continuous    |
| y      | Real   |     0   |        | input       | continuous    |
| Nd     | Real   |    10   |        | parameter   | fixed         |
| Td     | Real   |     0.1 |        | parameter   | fixed         |
| Ti     | Real   |     0.5 |        | parameter   | fixed         |
| k      | Real   |    20   |        | parameter   | fixed         |
| uMax   | Real   |     1   |        | parameter   | fixed         |
| uMin   | Real   |    -1   |        | parameter   | fixed         |



In [9]:
t0, tf, h = 0, 2, 0.01

print(f"Simulation start time: {t0} s")
print(f"Simulation end time:   {tf} s")
print(f"Simulation step size:  {h}  s")

Simulation start time: 0 s
Simulation end time:   2 s
Simulation step size:  0.01  s


In [10]:
from SysSimX.components.opensim_pendulum import OpenSimPendulum
from SysSimX.fem.pendulum import FEMPendulum

# Instantiate Pendulum Models (Modelica FMU, OpenSim, FEM/NGSolve)
eqb_pendulum  = fmu_models["pendulum"]
os_pendulum   = OpenSimPendulum()
fem_pendulum  = FEMPendulum()

# Wall contact angle
q_wall = fem_pendulum.geom_params.q_wall_deg

# Get mass and inertia from FEM model
m_pendulum       = fem_pendulum.mass
inertia_pendulum = fem_pendulum.inertia


# Compute length from inertia
# I = m*r^2  => with r = L => I = m*L^2  =>  L = sqrt(I/m)
L_computed = np.sqrt(inertia_pendulum / m_pendulum)

# Set parameters for FMU model
eqb_pendulum.parameters["m"].start = m_pendulum
eqb_pendulum.parameters["L"].start = L_computed

# Set parameters for OpenSim model
os_pendulum.mass = m_pendulum
os_pendulum.length = L_computed

# Set initial conditions
q0_deg = q_mean_deg
omega0 = 0

fem_pendulum.sim_params.t_end = tf
fem_pendulum.init_params.angular_position_deg = q0_deg
fem_pendulum.init_params.angular_velocity     = omega0
fem_pendulum.init_params.angular_acceleration = 0

eqb_pendulum.parameters['q0'].start     = np.deg2rad(q0_deg)
eqb_pendulum.parameters['omega0'].start = omega0

os_pendulum.q0     = np.deg2rad(q0_deg)
os_pendulum.omega0 = omega0


print(f"{'Wall Contact Angle:':<28} {q_wall:.2f} deg")
print(f"{'Pendulum Mass:':<28} {m_pendulum:.4f} kg")
print(f"{'Pendulum Inertia:':<28} {inertia_pendulum:.6f} kg·m²")
print(f"{'Pendulum Length (computed):':<28} {L_computed:.6f} m")

Wall Contact Angle:          0.00 deg
Pendulum Mass:               15.4458 kg
Pendulum Inertia:            7.000130 kg·m²
Pendulum Length (computed):  0.673206 m


**5. Initialize all components**

In [11]:
for component in [ref, sensor_ref, sensor_state, pid, drive, eqb_pendulum, os_pendulum, fem_pendulum]:
    component.initialize(t0)

fem_pendulum.visualize_state()

# Combine models in a dictionary for easy access
state_models = {"EQB": eqb_pendulum,
                "OpenSim": os_pendulum,
                "FEM": fem_pendulum}

LOG_SOLVER        | info    | CVODE linear multistep method CV_BDF
LOG_SOLVER        | info    | CVODE maximum integration order CV_ITER_NEWTON
LOG_SOLVER        | info    | CVODE use equidistant time grid YES
LOG_SOLVER        | info    | CVODE Using relative error tolerance 1.000000e-06
LOG_SOLVER        | info    | CVODE Using dense internal linear solver SUNLinSol_Dense.
LOG_SOLVER        | info    | CVODE Use internal dense numeric jacobian method.
LOG_SOLVER        | info    | CVODE uses internal root finding method NO
LOG_SOLVER        | info    | CVODE maximum absolut step size 0
LOG_SOLVER        | info    | CVODE initial step size is set automatically
LOG_SOLVER        | info    | CVODE maximum integration order 5
LOG_SOLVER        | info    | CVODE maximum number of nonlinear convergence failures permitted during one step 10
LOG_SOLVER        | info    | CVODE BDF stability limit detection algorithm OFF
LOG_SOLVER        | info    | CVODE linear multistep method CV_BDF
LOG_S

### Displacement in $m$

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

### Velocity in $\frac{m}{s}$

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

### Angular Acceleration in $\frac{m}{s}$

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

**6. Run the Co-Simulation with Logger**

In [12]:
# Logger
rows = []
def log(t, signals):
    rows.append({"t": t, **signals})
logger = log

# Start with OpenSim model
plant = state_models['FEM']
mode = "OpenSim"
q_tol = np.deg2rad(0.05)       
q_wall_rad = np.deg2rad(q_wall)

def determine_model(q_state, current_mode):
    angular_diff = q_state - q_wall_rad
    if current_mode == "OpenSim":
         if angular_diff < q_tol:
             return "FEM"
         else:
             return "OpenSim"
    elif current_mode == "FEM":
         if angular_diff > q_tol:
             return "OpenSim"
         else:
             return "FEM"

def sync_models(t, current_model, new_model):
    current_outputs = current_model.get_outputs()
    q_current = current_outputs['q']
    omega_current = current_outputs['omega']
    new_model.reinitialize(t, q_current, omega_current)

state_models['FEM'].initialize_scene()

### Pendulum Simulation: Stress and Displacement

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Text(value='Mode: ')

Text(value='Time: ')

Text(value='q_ref: ')

Text(value='q: \tomega: ')

Text(value='Torque: ')

In [13]:
import time

t = t0
while t < tf:
        # 1) Get reference signal
        ref.step(t, h)
        q_ref = ref.get_outputs()['q_ref']

        # 2) Get plant outputs at current time
        plant_out = plant.get_outputs()
        q_state, omega_state = plant_out['q'], plant_out['omega']

        # 3) Determine which model to use
        # new_mode = determine_model(q_state, mode)
        # if new_mode != mode:
        #      sync_models(t, plant, state_models[new_mode])
        #      plant = state_models[new_mode]
        #      mode = new_mode            

        # 4) Feed reference and state to sensor models
        sensor_ref.set_inputs(q=q_ref); 
        sensor_state.set_inputs(q=q_state)
        
        sensor_ref.step(t, h)
        sensor_state.step(t, h)

        U_q_ref = sensor_ref.get_outputs()['U_q']
        U_q_state = sensor_state.get_outputs()['U_q']

        # 5) Feed sensor outputs to PID controller
        pid.set_inputs(ref=U_q_ref, y=U_q_state)
        pid.step(t, h)
        u_pid = pid.get_outputs()['u']

        # 6) Drive
        drive.set_inputs(u_control=u_pid, omega=omega_state)
        drive.step(t, h)
        torque = drive.get_outputs()['torque']

        state = (t, q_ref, q_state, omega_state, torque, mode)

        # 6) Apply torque to plant and step
        plant.set_inputs(torque=torque)
        if isinstance(plant, FEMPendulum):
             plant.step(t, h, q_ref)
        else:
            plant.step(t, h)
            state_models['FEM'].update_scene(state)
            # wait a bit to see the update
            time.sleep(0.01)

        if logger:
            logger(t, dict(
                q_ref=q_ref, q_state=q_state, omega_state=omega_state,
                U_q_ref=U_q_ref, U_q_state=U_q_state,
                u_pid=u_pid, torque=torque
            ))
        
        t += h

**7. Plot the results**

In [15]:
import numpy as np
t_vals = [row['t'] for row in rows]
q_ref_vals = [row['q_ref'] for row in rows]
q_state_vals = [row['q_state'] for row in rows]
omega_state_vals = [row['omega_state'] for row in rows]

# Create a simple plotly figure for the q_ref and q_state over time
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_vals, y=q_ref_vals, mode='lines',
                            name='Reference Angle (q_ref)', line=dict(color='white', dash='dash')))
fig.add_trace(go.Scatter(x=t_vals, y=q_state_vals, mode='lines',
                            name='State Angle (q_state)', line=dict(color='red')))
fig.update_layout(
    title={'text': 'Pendulum Angle - FMUs + OpenSim Pendulum', 'font': {'size': 24}},
    xaxis_title={'text': 'Time (s)', 'font': {'size': 20}},
    yaxis_title={'text': 'Angle (rad)', 'font': {'size': 20}},
    legend_title={'text': 'Legend', 'font': {'size': 18}},
    font={'size': 16},
    template='plotly_dark'
)
# save the figure

fig.show()

**8. Export results to OpenSim MOT file**

In [ ]:
from SysSimX.utilities.results_opensim import create_opensim_mot_file

data = {'q': np.array([row['q_state'] for row in rows]),
        '/jointset/head_joint/q/speed': np.array(omega_state_vals)}
time = np.array([row['t'] for row in rows])
n_time_steps = time.shape[0]
filename = 'OpenSim/Results/demo_2_5.mot'

create_opensim_mot_file(data=data, time=time, filename=filename)